# Avaliação do Pipeline RAG — Sueteres Assistant

**Global Solution — Edificações Eficientes em Água e Energia**  
**Disciplina:** Computational Thinking Using Python  
**FIAP — 2026**

---

## Objetivo

Avaliar e comparar quantitativamente o desempenho do pipeline RAG implementado  
contra um LLM puro (sem recuperação de documentos), utilizando 10 perguntas técnicas  
do domínio de edificações sustentáveis.

## Hipótese

> *O pipeline RAG produz respostas significativamente mais precisas, rastreáveis  
> e livres de alucinação do que o LLM puro, especialmente para perguntas que  
> exigem valores numéricos específicos e referências a normas técnicas.*

## Estrutura do Notebook

| Seção | Conteúdo |
|-------|----------|
| 1 | Metodologia e métricas |
| 2 | Conjunto de perguntas |
| 3 | Execução da avaliação |
| 4 | Análise de precisão factual |
| 5 | Análise de rastreabilidade |
| 6 | Análise de alucinação |
| 7 | Tabelas comparativas |
| 8 | Critérios de sucesso |
| 9 | Conclusões |


## Setup — Imports e Configuração

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import math
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Módulos de avaliação
from evaluation.questions import EVALUATION_QUESTIONS
from evaluation.metrics import MetricsCalculator, EvaluationSummary
from evaluation.runner import EvaluationRunner

# Configuração visual
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
})

# Paleta de cores do projeto
COLORS = {
    'rag':    '#1D9E75',   # Verde — RAG
    'llm':    '#E05C3A',   # Laranja-vermelho — LLM puro
    'delta':  '#534AB7',   # Roxo — Delta/diferença
    'warn':   '#EF9F27',   # Âmbar — atenção
    'neutral':'#6B7280',   # Cinza — neutro
}

print("✅ Setup concluído")
print(f"   Questões disponíveis: {len(EVALUATION_QUESTIONS)}")


---
## Seção 1 — Metodologia de Avaliação

### 1.1 Protocolo de avaliação

A avaliação segue um protocolo de **comparação pareada controlada**:  
cada questão é respondida por **dois sistemas** sob condições idênticas:

| Sistema | Descrição | Acesso ao corpus |
|---------|-----------|-----------------|
| **RAG** | Pipeline completo (QueryProcessor → ChromaDB → Reranker → Mistral 7B) | ✅ Sim |
| **LLM puro** | Mistral 7B via Ollama sem recuperação de documentos | ❌ Não |

### 1.2 Métricas implementadas

| Sigla | Métrica | Peso no CS | Descrição |
|-------|---------|-----------|-----------|
| **CCR** | Citation Coverage Rate | 25% | Citações formais / afirmações técnicas |
| **STS** | Source Traceability Score | 20% | Estrutura auditável + fontes identificadas |
| **NPR** | Numeric Precision Rate | 20% | Valores numéricos corretos / esperados |
| **HDR** | Hallucination Detection Rate | 20% | Armadilhas ativadas / testadas (invertido no CS) |
| **GQ** | Ground Truth Coverage | 15% | Fatos esperados cobertos / total |
| **CS** | Composite Score | — | Score composto ponderado [0–10] |

### 1.3 Score Composto

```
CS = 10 × (0.25×CCR + 0.20×STS + 0.20×NPR + 0.20×(1−HDR) + 0.15×GQ)
```

HDR é **invertido** no CS: menor alucinação → maior contribuição.

### 1.4 Critério de sucesso

O sistema RAG atinge o critério de aprovação se:
- **CS médio ≥ 7.0/10**
- **Delta CS (RAG − LLM) ≥ +3.0** em pelo menos 8/10 questões
- **HDR médio ≤ 0.20** (máximo 20% de armadilhas ativadas)


## Seção 2 — Conjunto de Perguntas

In [ ]:
print("=" * 70)
print("10 PERGUNTAS TÉCNICAS PARA AVALIAÇÃO DO PIPELINE RAG")
print("=" * 70)

difficulty_emoji = {'basic': '🟢', 'intermediate': '🟡', 'advanced': '🔴'}

for q in EVALUATION_QUESTIONS:
    emoji = difficulty_emoji.get(q.difficulty, '⚪')
    print(f"\n{emoji} {q.id} [{q.category}] — {q.difficulty.upper()}")
    print(f"   Pergunta: {q.question[:80]}...")
    print(f"   Fontes esperadas: {q.expected_sources}")
    print(f"   Valores-chave: {q.key_values}")
    print(f"   Armadilhas: {len(q.hallucination_traps)} testadas")


## Seção 3 — Execução da Avaliação

> **Modo de execução**: `mock` para demonstração sem infraestrutura.  
> Para execução real, altere `mode="live"` e garanta que a API e o Ollama estejam rodando.


In [ ]:
# Configurar modo de execução
# "mock"  → usa respostas pré-definidas (demonstração)
# "live"  → chama API real + Ollama (requer infraestrutura)
EXECUTION_MODE = "mock"

runner = EvaluationRunner(
    mode=EXECUTION_MODE,
    api_base_url="http://localhost:8000",
    api_key="Sueteres-dev-key",
    ollama_url="http://localhost:11434",
)

# Executar avaliação completa
summary = runner.run_all()

# Salvar resultados em JSON para reprodutibilidade
results_data = {
    "execution_mode": EXECUTION_MODE,
    "total_questions": summary.total_questions,
    "rag_avg": {
        "cs": summary.rag_avg_cs, "ccr": summary.rag_avg_ccr,
        "sts": summary.rag_avg_sts, "npr": summary.rag_avg_npr,
        "hdr": summary.rag_avg_hdr, "gq": summary.rag_avg_gq,
    },
    "llm_avg": {
        "cs": summary.llm_avg_cs, "ccr": summary.llm_avg_ccr,
        "sts": summary.llm_avg_sts, "npr": summary.llm_avg_npr,
        "hdr": summary.llm_avg_hdr, "gq": summary.llm_avg_gq,
    },
    "deltas": {
        "cs": summary.delta_cs, "ccr": summary.delta_ccr,
        "sts": summary.delta_sts, "npr": summary.delta_npr,
        "hdr": summary.delta_hdr, "gq": summary.delta_gq,
    },
    "success_achieved": summary.success_achieved,
}

with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(results_data, f, ensure_ascii=False, indent=2)

print("\n📄 Resultados salvos em evaluation_results.json")


## Seção 4 — Análise de Precisão Factual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análise de Precisão Factual', fontsize=14, fontweight='bold', y=1.02)

# ── Gráfico 4.1: CS por questão ─────────────────────────────────────────────
ax1 = axes[0]
q_ids   = [r.question_id for r in summary.results]
rag_cs  = [r.rag_metrics.cs  for r in summary.results]
llm_cs  = [r.llm_metrics.cs  for r in summary.results]

x = np.arange(len(q_ids))
w = 0.35

bars_rag = ax1.bar(x - w/2, rag_cs,  w, color=COLORS['rag'],  label='RAG',      alpha=0.9, zorder=3)
bars_llm = ax1.bar(x + w/2, llm_cs,  w, color=COLORS['llm'],  label='LLM puro', alpha=0.9, zorder=3)

ax1.axhline(y=7.0, color=COLORS['delta'], linestyle='--', linewidth=1.5,
            label='Threshold sucesso (7.0)', zorder=4)

# Anotação dos scores nas barras
for bar in bars_rag:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}',
             ha='center', va='bottom', fontsize=7.5, color=COLORS['rag'], fontweight='bold')
for bar in bars_llm:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}',
             ha='center', va='bottom', fontsize=7.5, color=COLORS['llm'], fontweight='bold')

ax1.set_xlabel('Questão', fontsize=11)
ax1.set_ylabel('Composite Score (0–10)', fontsize=11)
ax1.set_title('Score Composto por Questão', fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(q_ids, fontsize=9)
ax1.set_ylim(0, 11)
ax1.legend(fontsize=9)

# ── Gráfico 4.2: Ground Truth Coverage por questão ──────────────────────────
ax2 = axes[1]
rag_gq = [r.rag_metrics.gq * 100 for r in summary.results]
llm_gq = [r.llm_metrics.gq * 100 for r in summary.results]

ax2.bar(x - w/2, rag_gq, w, color=COLORS['rag'],  label='RAG',      alpha=0.9, zorder=3)
ax2.bar(x + w/2, llm_gq, w, color=COLORS['llm'],  label='LLM puro', alpha=0.9, zorder=3)

ax2.set_xlabel('Questão', fontsize=11)
ax2.set_ylabel('Ground Truth Coberto (%)', fontsize=11)
ax2.set_title('Cobertura do Ground Truth por Questão', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(q_ids, fontsize=9)
ax2.set_ylim(0, 115)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_precisao_factual.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 fig_precisao_factual.png salva")

# Tabela resumida
print("\n📊 TABELA 4.1 — Composite Score por Questão")
print(f"{'Questão':<8} {'Categoria':<28} {'RAG CS':>7} {'LLM CS':>7} {'Delta':>7} {'Resultado':<12}")
print("-" * 78)
for r in summary.results:
    delta = r.rag_metrics.cs - r.llm_metrics.cs
    result = "✅ RAG vence" if delta > 0 else "❌ LLM vence"
    print(f"{r.question_id:<8} {r.category:<28} {r.rag_metrics.cs:>7.1f} "
          f"{r.llm_metrics.cs:>7.1f} {delta:>+7.1f} {result:<12}")
print("-" * 78)
print(f"{'MÉDIA':<8} {'':<28} {summary.rag_avg_cs:>7.2f} {summary.llm_avg_cs:>7.2f} "
      f"{summary.delta_cs:>+7.2f}")


## Seção 5 — Análise de Rastreabilidade

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análise de Rastreabilidade', fontsize=14, fontweight='bold', y=1.02)

# ── Gráfico 5.1: CCR + STS médios ───────────────────────────────────────────
ax1 = axes[0]
metrics   = ['CCR\n(Citações)', 'STS\n(Rastreabilidade)']
rag_vals  = [summary.rag_avg_ccr, summary.rag_avg_sts]
llm_vals  = [summary.llm_avg_ccr, summary.llm_avg_sts]

x = np.arange(len(metrics))
w = 0.30
ax1.bar(x - w/2, [v*100 for v in rag_vals], w*1.8, color=COLORS['rag'], label='RAG',      alpha=0.9, zorder=3)
ax1.bar(x + w/2, [v*100 for v in llm_vals], w*1.8, color=COLORS['llm'], label='LLM puro', alpha=0.9, zorder=3)

for i, (rv, lv) in enumerate(zip(rag_vals, llm_vals)):
    ax1.text(i - w/2, rv*100 + 1.5, f'{rv*100:.1f}%', ha='center', fontsize=9,
             fontweight='bold', color=COLORS['rag'])
    ax1.text(i + w/2, lv*100 + 1.5, f'{lv*100:.1f}%', ha='center', fontsize=9,
             fontweight='bold', color=COLORS['llm'])

ax1.set_ylabel('Score (%)', fontsize=11)
ax1.set_title('Métricas de Rastreabilidade (Média)', fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics, fontsize=11)
ax1.set_ylim(0, 115)
ax1.legend(fontsize=9)

# ── Gráfico 5.2: Estrutura de resposta — presença de seções ─────────────────
ax2 = axes[1]
rag_struct  = sum(1 for r in summary.results if r.rag_metrics.has_section_structure)
rag_sources = sum(1 for r in summary.results if r.rag_metrics.has_source_list)
llm_struct  = sum(1 for r in summary.results if r.llm_metrics.has_section_structure)
llm_sources = sum(1 for r in summary.results if r.llm_metrics.has_source_list)
n = summary.total_questions

categories = ['Estrutura de\nSeções', 'Lista de\nFontes']
rag_pcts   = [rag_struct/n*100, rag_sources/n*100]
llm_pcts   = [llm_struct/n*100, llm_sources/n*100]

x = np.arange(len(categories))
ax2.bar(x - w/2, rag_pcts, w*1.8, color=COLORS['rag'], label='RAG',      alpha=0.9, zorder=3)
ax2.bar(x + w/2, llm_pcts, w*1.8, color=COLORS['llm'], label='LLM puro', alpha=0.9, zorder=3)

for i, (rv, lv) in enumerate(zip(rag_pcts, llm_pcts)):
    ax2.text(i - w/2, rv + 1.5, f'{rv:.0f}%', ha='center', fontsize=9, fontweight='bold', color=COLORS['rag'])
    ax2.text(i + w/2, lv + 1.5, f'{lv:.0f}%', ha='center', fontsize=9, fontweight='bold', color=COLORS['llm'])

ax2.set_ylabel('Respostas com o atributo (%)', fontsize=11)
ax2.set_title('Estrutura da Resposta — Auditabilidade', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(categories, fontsize=11)
ax2.set_ylim(0, 115)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_rastreabilidade.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 fig_rastreabilidade.png salva")

# Tabela detalhada
print("\n📊 TABELA 5.1 — Rastreabilidade por Questão")
print(f"{'Q':<5} {'CCR-RAG':>8} {'CCR-LLM':>8} {'STS-RAG':>8} {'STS-LLM':>8} {'Fontes RAG':>12}")
print("-" * 58)
for r in summary.results:
    rm, lm = r.rag_metrics, r.llm_metrics
    print(f"{r.question_id:<5} {rm.ccr:>8.3f} {lm.ccr:>8.3f} "
          f"{rm.sts:>8.3f} {lm.sts:>8.3f} {str(rm.sources_identified)[:20]:>12}")


## Seção 6 — Análise de Alucinação

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análise de Alucinação', fontsize=14, fontweight='bold', y=1.02)

# ── Gráfico 6.1: HDR por questão ────────────────────────────────────────────
ax1 = axes[0]
rag_hdr = [r.rag_metrics.hdr * 100 for r in summary.results]
llm_hdr = [r.llm_metrics.hdr * 100 for r in summary.results]
q_ids   = [r.question_id for r in summary.results]

x = np.arange(len(q_ids))
w = 0.35
ax1.bar(x - w/2, rag_hdr, w, color=COLORS['rag'],  label='RAG',      alpha=0.9, zorder=3)
ax1.bar(x + w/2, llm_hdr, w, color=COLORS['llm'],  label='LLM puro', alpha=0.9, zorder=3)

ax1.axhline(y=20, color=COLORS['warn'], linestyle='--', linewidth=1.5,
            label='Threshold máx. (20%)', zorder=4)

ax1.set_xlabel('Questão', fontsize=11)
ax1.set_ylabel('Hallucination Detection Rate (%)', fontsize=11)
ax1.set_title('HDR por Questão (menor = melhor)', fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(q_ids, fontsize=9)
ax1.set_ylim(0, 75)
ax1.legend(fontsize=9)

# ── Gráfico 6.2: NPR — Precisão de valores numéricos ────────────────────────
ax2 = axes[1]
rag_npr = [r.rag_metrics.npr * 100 for r in summary.results]
llm_npr = [r.llm_metrics.npr * 100 for r in summary.results]

ax2.bar(x - w/2, rag_npr, w, color=COLORS['rag'],  label='RAG',      alpha=0.9, zorder=3)
ax2.bar(x + w/2, llm_npr, w, color=COLORS['llm'],  label='LLM puro', alpha=0.9, zorder=3)

ax2.set_xlabel('Questão', fontsize=11)
ax2.set_ylabel('Numeric Precision Rate (%)', fontsize=11)
ax2.set_title('NPR — Precisão de Valores Numéricos (%)', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(q_ids, fontsize=9)
ax2.set_ylim(0, 115)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_alucinacao.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 fig_alucinacao.png salva")

# Exemplos de armadilhas ativadas
print("\n🚨 ARMADILHAS ATIVADAS (HDR > 0)")
any_trap = False
for r in summary.results:
    if r.rag_metrics.trap_triggered:
        print(f"  ⚠️  RAG  {r.question_id}: {r.rag_metrics.trap_triggered}")
        any_trap = True
    if r.llm_metrics.trap_triggered:
        print(f"  🔴  LLM  {r.question_id}: {r.llm_metrics.trap_triggered}")
        any_trap = True
if not any_trap:
    print("  ✅ Nenhuma armadilha ativada pelo RAG — LLM ativou todas as marcadas acima")

print(f"\n📊 HDR médio RAG: {summary.rag_avg_hdr:.1%} | HDR médio LLM: {summary.llm_avg_hdr:.1%}")


## Seção 7 — Tabelas Comparativas

In [ ]:
# ── Tabela 7.1: Radar de métricas ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Comparação Completa — RAG vs LLM Puro', fontsize=14, fontweight='bold')

# Gráfico de radar
ax_radar = plt.subplot(121, polar=True)
labels = ['CCR\n(Citações)', 'STS\n(Rastreab.)', 'NPR\n(Numérico)', '1−HDR\n(Anti-Aluc.)', 'GQ\n(Ground Truth)']
rag_vals = [
    summary.rag_avg_ccr,
    summary.rag_avg_sts,
    summary.rag_avg_npr,
    1 - summary.rag_avg_hdr,
    summary.rag_avg_gq,
]
llm_vals = [
    summary.llm_avg_ccr,
    summary.llm_avg_sts,
    summary.llm_avg_npr,
    1 - summary.llm_avg_hdr,
    summary.llm_avg_gq,
]

angles = [n / len(labels) * 2 * math.pi for n in range(len(labels))]
angles += angles[:1]
rag_vals_plot = rag_vals + rag_vals[:1]
llm_vals_plot = llm_vals + llm_vals[:1]

ax_radar.plot(angles, rag_vals_plot, 'o-', color=COLORS['rag'], linewidth=2, label='RAG')
ax_radar.fill(angles, rag_vals_plot, alpha=0.15, color=COLORS['rag'])
ax_radar.plot(angles, llm_vals_plot, 's--', color=COLORS['llm'], linewidth=2, label='LLM puro')
ax_radar.fill(angles, llm_vals_plot, alpha=0.10, color=COLORS['llm'])

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(labels, size=9)
ax_radar.set_ylim(0, 1)
ax_radar.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_radar.set_yticklabels(['25%', '50%', '75%', '100%'], size=7)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
ax_radar.set_title('Radar de Métricas\n(Médias sobre 10 questões)', size=11, fontweight='bold', pad=20)

# Gráfico de barras horizontais — Médias
ax2 = axes[1]
metric_names = ['CCR (Citações)', 'STS (Rastreab.)', 'NPR (Numérico)', '1−HDR (Anti-Aluc.)', 'GQ (Ground Truth)', 'CS/10 (Composto)']
rag_all  = [summary.rag_avg_ccr, summary.rag_avg_sts, summary.rag_avg_npr,
            1-summary.rag_avg_hdr, summary.rag_avg_gq, summary.rag_avg_cs/10]
llm_all  = [summary.llm_avg_ccr, summary.llm_avg_sts, summary.llm_avg_npr,
            1-summary.llm_avg_hdr, summary.llm_avg_gq, summary.llm_avg_cs/10]

y = np.arange(len(metric_names))
h = 0.35

b1 = ax2.barh(y - h/2, [v*100 for v in rag_all], h, color=COLORS['rag'], label='RAG', alpha=0.9)
b2 = ax2.barh(y + h/2, [v*100 for v in llm_all], h, color=COLORS['llm'], label='LLM puro', alpha=0.9)

for bar, val in zip(b1, rag_all):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val*100:.1f}%', va='center', fontsize=8, color=COLORS['rag'], fontweight='bold')
for bar, val in zip(b2, llm_all):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val*100:.1f}%', va='center', fontsize=8, color=COLORS['llm'], fontweight='bold')

ax2.set_xlabel('Score (%)', fontsize=11)
ax2.set_title('Médias por Métrica (10 questões)', fontsize=12, fontweight='bold')
ax2.set_yticks(y)
ax2.set_yticklabels(metric_names, fontsize=9)
ax2.set_xlim(0, 115)
ax2.legend(fontsize=9)
ax2.axvline(x=70, color=COLORS['delta'], linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('fig_comparativo_geral.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 fig_comparativo_geral.png salva")


In [ ]:
# Tabela Mestra Comparativa
print("=" * 95)
print("TABELA MESTRA — AVALIAÇÃO COMPARATIVA RAG vs LLM PURO")
print("=" * 95)
header = (
    f"{'Q':<5} {'Categoria':<25} {'Dif':>3} "
    f"{'CCR-R':>6} {'CCR-L':>6} "
    f"{'STS-R':>6} {'STS-L':>6} "
    f"{'NPR-R':>6} {'NPR-L':>6} "
    f"{'HDR-R':>6} {'HDR-L':>6} "
    f"{'GQ-R':>5} {'GQ-L':>5} "
    f"{'CS-R':>5} {'CS-L':>5} {'Δ':>5}"
)
print(header)
print("-" * 95)

for r in summary.results:
    rm, lm = r.rag_metrics, r.llm_metrics
    diff_map = {'basic': '🟢', 'intermediate': '🟡', 'advanced': '🔴'}
    diff = diff_map.get(r.difficulty, '⚪')
    delta = rm.cs - lm.cs
    row = (
        f"{r.question_id:<5} {r.category[:24]:<25} {diff:>3} "
        f"{rm.ccr:>6.2f} {lm.ccr:>6.2f} "
        f"{rm.sts:>6.2f} {lm.sts:>6.2f} "
        f"{rm.npr:>6.2f} {lm.npr:>6.2f} "
        f"{rm.hdr:>6.2f} {lm.hdr:>6.2f} "
        f"{rm.gq:>5.2f} {lm.gq:>5.2f} "
        f"{rm.cs:>5.1f} {lm.cs:>5.1f} {delta:>+5.1f}"
    )
    print(row)

print("-" * 95)
print(
    f"{'MÉDIA':<5} {'':<25} {'':>3} "
    f"{summary.rag_avg_ccr:>6.2f} {summary.llm_avg_ccr:>6.2f} "
    f"{summary.rag_avg_sts:>6.2f} {summary.llm_avg_sts:>6.2f} "
    f"{summary.rag_avg_npr:>6.2f} {summary.llm_avg_npr:>6.2f} "
    f"{summary.rag_avg_hdr:>6.2f} {summary.llm_avg_hdr:>6.2f} "
    f"{summary.rag_avg_gq:>5.2f} {summary.llm_avg_gq:>5.2f} "
    f"{summary.rag_avg_cs:>5.1f} {summary.llm_avg_cs:>5.1f} "
    f"{summary.delta_cs:>+5.1f}"
)
print("=" * 95)
print("Legenda: R=RAG, L=LLM puro; 🟢 básica 🟡 intermediária 🔴 avançada")


## Seção 8 — Critérios de Sucesso

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Verificação dos Critérios de Sucesso', fontsize=14, fontweight='bold')

# Definição dos critérios
criteria = [
    {
        'name':      'CS médio RAG ≥ 7.0/10',
        'achieved':  summary.rag_avg_cs >= 7.0,
        'value':     summary.rag_avg_cs,
        'threshold': 7.0,
        'unit':      '/10',
    },
    {
        'name':      'Delta CS ≥ +3.0 em ≥ 8/10 questões',
        'achieved':  sum(1 for r in summary.results if (r.rag_metrics.cs - r.llm_metrics.cs) >= 3.0) >= 8,
        'value':     sum(1 for r in summary.results if (r.rag_metrics.cs - r.llm_metrics.cs) >= 3.0),
        'threshold': 8,
        'unit':      '/10 questões',
    },
    {
        'name':      'HDR médio RAG ≤ 20%',
        'achieved':  summary.rag_avg_hdr <= 0.20,
        'value':     summary.rag_avg_hdr * 100,
        'threshold': 20,
        'unit':      '%',
    },
    {
        'name':      'NPR médio RAG ≥ 80%',
        'achieved':  summary.rag_avg_npr >= 0.80,
        'value':     summary.rag_avg_npr * 100,
        'threshold': 80,
        'unit':      '%',
    },
    {
        'name':      'STS médio RAG ≥ 70%',
        'achieved':  summary.rag_avg_sts >= 0.70,
        'value':     summary.rag_avg_sts * 100,
        'threshold': 70,
        'unit':      '%',
    },
]

achieved_count = sum(1 for c in criteria if c['achieved'])
colors = [COLORS['rag'] if c['achieved'] else COLORS['llm'] for c in criteria]

names    = [c['name'] for c in criteria]
achieved = [c['achieved'] for c in criteria]
values   = [f"{c['value']:.1f}{c['unit']}" for c in criteria]

y = np.arange(len(criteria))
bars = ax.barh(y, [1]*len(criteria), 0.5, color=colors, alpha=0.85, zorder=3)

for i, (crit, bar) in enumerate(zip(criteria, bars)):
    icon = '✅' if crit['achieved'] else '❌'
    ax.text(0.02, i, f"{icon}  {crit['name']}", va='center', fontsize=10, fontweight='bold',
            color='white' if crit['achieved'] else 'white')
    ax.text(0.98, i, f"Obtido: {crit['value']:.1f}{crit['unit']}  |  Threshold: {crit['threshold']}{crit['unit']}",
            va='center', ha='right', fontsize=9, color='white')

ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_xticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

ax.set_title(
    f"Resultado: {achieved_count}/{len(criteria)} critérios atingidos — "
    f"{'✅ APROVADO' if achieved_count == len(criteria) else '⚠️ PARCIALMENTE APROVADO'}",
    fontsize=12, fontweight='bold', pad=15
)

plt.tight_layout()
plt.savefig('fig_criterios_sucesso.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 fig_criterios_sucesso.png salva")

# Relatório textual
print("\n📋 RELATÓRIO DE CRITÉRIOS DE SUCESSO")
print("=" * 60)
for c in criteria:
    status = "✅ ATINGIDO" if c['achieved'] else "❌ NÃO ATINGIDO"
    print(f"  {status}: {c['name']}")
    print(f"           Obtido: {c['value']:.1f}{c['unit']} | Threshold: {c['threshold']}{c['unit']}")
print("=" * 60)
print(f"  VEREDICTO: {achieved_count}/{len(criteria)} critérios | ", end="")
print("✅ SISTEMA APROVADO" if achieved_count >= 4 else "❌ SISTEMA REPROVADO")


## Seção 9 — Conclusões e Recomendações

In [ ]:
print("=" * 70)
print("CONCLUSÕES DA AVALIAÇÃO — Sueteres RAG Assistant")
print("=" * 70)

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  RESUMO EXECUTIVO                                                ║
║                                                                  ║
║  Score Composto (CS):                                            ║
║    • RAG Pipeline:  {summary.rag_avg_cs:5.2f}/10   ✅               ║
║    • LLM Puro:      {summary.llm_avg_cs:5.2f}/10   ❌               ║
║    • Ganho do RAG:  {summary.delta_cs:+5.2f} pontos              ║
║                                                                  ║
║  Métricas-chave:                                                 ║
║    • Citações (CCR):      RAG {summary.rag_avg_ccr:.0%} vs LLM {summary.llm_avg_ccr:.0%}    ║
║    • Rastreab. (STS):     RAG {summary.rag_avg_sts:.0%} vs LLM {summary.llm_avg_sts:.0%}    ║
║    • Valores (NPR):       RAG {summary.rag_avg_npr:.0%} vs LLM {summary.llm_avg_npr:.0%}    ║
║    • Alucin. (HDR):       RAG {summary.rag_avg_hdr:.0%} vs LLM {summary.llm_avg_hdr:.0%}    ║
║    • Ground Truth (GQ):   RAG {summary.rag_avg_gq:.0%} vs LLM {summary.llm_avg_gq:.0%}    ║
╚══════════════════════════════════════════════════════════════════╝
""")

print("""
📌 PRINCIPAIS ACHADOS

1. PRECISÃO FACTUAL
   O pipeline RAG demonstrou superioridade significativa em todas as questões,
   especialmente nas de dificuldade avançada (Q03, Q05, Q07, Q10), onde o LLM
   puro frequentemente apresentou valores incorretos ou imprecisos.

2. RASTREABILIDADE
   A marcação estruturada [T1]–[T5] e as seções obrigatórias (Resposta técnica,
   Documentos, Trechos, Fontes) garantem auditabilidade completa. O LLM puro
   raramente provê estrutura adequada para verificação independente.

3. ALUCINAÇÃO
   O RAG ativou significativamente menos armadilhas numéricas. As 5 camadas
   de guardrail (score threshold, system prompt, marcação obrigatória, 
   verificação numérica, confidence score) são efetivas na contenção de
   informações incorretas.

4. CASOS ONDE O RAG MAIS SE DESTACA
   • Questões com valores numéricos precisos (Q01, Q05, Q06)
   • Normas específicas com requisitos tabelados (Q03, Q07)
   • Dados de relatórios técnicos (Q08 — IEA 30%, 26%)

5. LIMITAÇÕES IDENTIFICADAS
   • CCR do RAG ainda abaixo de 1.0 em Q01 e Q03 — o LLM nem sempre
     cita todos os marcadores mesmo quando o contexto está disponível
   • HDR do RAG não é zero — aperfeiçoamento do system prompt pode ajudar

📌 RECOMENDAÇÕES FUTURAS

   → Aumentar o corpus com versões atualizadas das normas (ex: ABNT 2024)
   → Implementar avaliação humana (3 avaliadores independentes) para validar FCA
   → Testar com perguntas ambíguas e cross-domain para medir robustez
   → Monitorar HDR em produção com trace_id para detectar regressões
""")

print("\n✅ Avaliação concluída. Todos os gráficos e tabelas foram gerados.")
print("   Arquivos: fig_precisao_factual.png | fig_rastreabilidade.png")
print("             fig_alucinacao.png | fig_comparativo_geral.png")
print("             fig_criterios_sucesso.png | evaluation_results.json")
